<a href="https://colab.research.google.com/github/Yanina2105/MD-Lab14_FT/blob/develop/LABORATORIO_14_CONCEPTOS_PRELIMINARES_MULTICLASIFICADORES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SEMANA 14: CONCEPTOS PRELIMINARES. MULTICLASIFICADORES**

1. Tomando la información disponible en el repositorio UCI Machine Learning
https://archive.ics.uci.edu/ml/datasets/Productivity+Prediction+of+Garment+Employees, pero
eliminando previamente a la variable ‘date’ y tomando a la variable ‘actual_productivity’. En este caso se estará usando este link: https://raw.githubusercontent.com/LuYuChen03/Complex-system/main/garments_worker_productivity.csv

In [1]:
import pandas as pd
import numpy as np

In [2]:
url = 'https://raw.githubusercontent.com/LuYuChen03/Complex-system/main/garments_worker_productivity.csv'
df = pd.read_csv(url)
df.head()

,year,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,7/12/1905,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382


## **a. Realice el preprocesamiento de la información que incluya el análisis de datos faltantes y tratamiento de outliers a nivel univariado y multivariado. Además, convierta las variables categóricas a dummies y aplique un escalamiento a las variables numéricas.**

In [3]:
df.drop('year', axis=1, inplace=True)

In [4]:
df.isnull().sum()

,0
quarter,0
department,0
day,0
team,0
targeted_productivity,0
smv,0
wip,506
over_time,0
incentive,0
idle_time,0


In [5]:
df.dtypes

,0
quarter,object
department,object
day,object
team,int64
targeted_productivity,float64
smv,float64
wip,float64
over_time,int64
incentive,int64
idle_time,float64


In [6]:
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
num_cols

Index(['team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive',
       'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers',
       'actual_productivity'],
      dtype='object')

In [7]:
cat_cols = df.select_dtypes(include=['object']).columns
cat_cols

Index(['quarter', 'department', 'day'], dtype='object')

In [8]:
Q1 = df[num_cols].quantile(0.25)
Q3 = df[num_cols].quantile(0.75)
IQR = Q3 - Q1
df = df[~((df[num_cols] < (Q1 - 1.5 * IQR)) | (df[num_cols] > (Q3 + 1.5 * IQR))).any(axis=1)]

In [10]:
from sklearn.impute import SimpleImputer

In [11]:
imputer = SimpleImputer(strategy='mean')
df[num_cols] = imputer.fit_transform(df[num_cols])

In [12]:
df = pd.get_dummies(df, columns=['quarter', 'department', 'day'], drop_first=True)

In [13]:
X = df.drop('actual_productivity', axis=1)
y = df['actual_productivity']

In [14]:
from sklearn.preprocessing import StandardScaler

In [15]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## **b. Separe los datos en entrenamiento (80%) y prueba (20%) y, realice un ensamblaje utilizando las técnicas de voting, bagging, boosting y stacking para los modelos k-NN, SVM, regresión lineal, árbol de clasificación y Random Forest definiendo distintos hiperparámetros para cada modelo.**

In [16]:
from sklearn.model_selection import train_test_split

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

In [18]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [19]:
models = {
    'knn': KNeighborsRegressor(n_neighbors=3),
    'svm': SVR(C=10, kernel='rbf'),
    'lr': LinearRegression(),
    'tree': DecisionTreeRegressor(max_depth=6, random_state=42),
    'rf': RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
}

In [27]:
from sklearn.ensemble import VotingRegressor

In [28]:
voting = VotingRegressor(estimators=[(name, model) for name, model in models.items()])

In [29]:
from sklearn.ensemble import BaggingRegressor

In [35]:
bagging = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=6),
    n_estimators=10,
    random_state=42
)

In [36]:
from sklearn.ensemble import GradientBoostingRegressor

In [37]:
boosting = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

In [38]:
from sklearn.ensemble import StackingRegressor

In [41]:
stacking = StackingRegressor(
    estimators=[(name, model) for name, model in models.items()],
    final_estimator=LinearRegression()
)

In [43]:
ensemble_models = {
    'Voting': voting,
    'Bagging': bagging,
    'Boosting': boosting,
    'Stacking': stacking
}

## **c. Utilice las métricas error cuadrático medio, raíz del error cuadrático medio, error absoluto medio y coeficiente de determinación para elegir el mejor modelo que prediga a la variable ‘actual_productivity’, justificando su respuesta.**

In [44]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [45]:
results = {}

In [46]:
for name, model in ensemble_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results[name] = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2
    }

In [48]:
results_df = pd.DataFrame(results).T.sort_values(by='RMSE')
print("Resultados ordenados por mejor RMSE:\n")
print(results_df)

Resultados ordenados por mejor RMSE:

               MSE      RMSE       MAE        R²
Boosting  0.011908  0.109126  0.069204  0.331534
Bagging   0.012334  0.111057  0.074533  0.307666
Stacking  0.012508  0.111839  0.073375  0.297879
Voting    0.012949  0.113792  0.077452  0.273147


**El mejor modelo es el que tenga menor RMSE y mayor R². Boosting (GradientBoostingRegressor) sería el modelo recomendado por su precisión y generalización.**